### SECTION 1: Import Dependencies

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when
from pyspark.sql.types import *
import pandas as pd
from sqlalchemy import create_engine
from typing import Optional


### SECTION 2: Configuration

In [3]:
DATA_PATH = r"C:\Users\USER\Desktop\Data_Recap\pyspark\etl_nuga_bank\rawdata\nuga_bank_transactions.csv"
APP_NAME = "NugaBankETL"
# POSTGRES_DB = "..."
# POSTGRES_USER = "..."

### SECTION 3: Spark Session

In [4]:
# Initialise the spark session
spark = (
    SparkSession.builder
    .appName('NugaBankETL')
    .master('local[*]')
    .getOrCreate()
)

### SECTION 4: Data Extraction / Loading the csv 

In [5]:
nuga_df = (
    spark.read
        .option("header", True) # this means use the first row as column names
        .option("inferSchema", True) # this scans the file, it notices numbers, so creates integerTyp or double, depending on datatype
        .csv(DATA_PATH)
)

### SECTION 5: Data Profiling

In [6]:
# nuga_df.show(10, truncate=False)
# nuga_df.printSchema()
# nuga_df.columns
# nuga_df.count()
nuga_df.describe().show()


+-------+------------------+----------------+-------------+--------------------+-------------+--------------+----------------+-------------+------------------+-------------------+--------------------+--------------------+--------------------+-------------+------------------+--------+------+---------+--------------------+------+--------------+
|summary|            Amount|Transaction_Type|Customer_Name|    Customer_Address|Customer_City|Customer_State|Customer_Country|      Company|         Job_Title|              Email|        Phone_Number|  Credit_Card_Number|                IBAN|Currency_Code|     Random_Number|Category| Group|Is_Active|         Description|Gender|Marital_Status|
+-------+------------------+----------------+-------------+--------------------+-------------+--------------+----------------+-------------+------------------+-------------------+--------------------+--------------------+--------------------+-------------+------------------+--------+------+---------+---------

In [7]:
nuga_df.selectExpr(
    "count(*) as total_rows",
    "count(distinct Customer_Name) as unique_customers"
).show()

+----------+----------------+
|total_rows|unique_customers|
+----------+----------------+
|   1000000|          312447|
+----------+----------------+



In [8]:
# Display how many null values each colum contains
# from pyspark.sql.functions import col, count, when

nuga_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in nuga_df.columns
]).show(vertical=True)

-RECORD 0--------------------
 Transaction_Date   | 0      
 Amount             | 0      
 Transaction_Type   | 0      
 Customer_Name      | 100425 
 Customer_Address   | 100087 
 Customer_City      | 100034 
 Customer_State     | 100009 
 Customer_Country   | 100672 
 Company            | 100295 
 Job_Title          | 99924  
 Email              | 100043 
 Phone_Number       | 100524 
 Credit_Card_Number | 100085 
 IBAN               | 100300 
 Currency_Code      | 99342  
 Random_Number      | 99913  
 Category           | 100332 
 Group              | 100209 
 Is_Active          | 100259 
 Last_Updated       | 100321 
 Description        | 100403 
 Gender             | 99767  
 Marital_Status     | 99904  



In [9]:
# this shows us exactly which values exist in Is_Active e.g (Yes, No, Null, etc)
nuga_df.groupBy("Is_Active").count().show()

+---------+------+
|Is_Active| count|
+---------+------+
|     NULL|100259|
|       No|449899|
|      Yes|449842|
+---------+------+



In [10]:
"""
this will tell us that the transaction types are consistent 
and don't contain issues such as different capitalization like:
(Deposit, deposit, DEPOSIT) or mispellings
""" 
nuga_df.groupBy("Transaction_Type").count().show()

+----------------+------+
|Transaction_Type| count|
+----------------+------+
|         Deposit|333120|
|        Transfer|333301|
|      Withdrawal|333579|
+----------------+------+



### SECTION 6: Data Cleaning

In [11]:
# Step 1 - Import dependencies
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, when, count

def summarize_missing_values(df: DataFrame) -> DataFrame:
    """
    Generate a summary of missing (NULL) values for every column in a Spark DataFrame

    Parameter
    ---------
        df : DataFrame 
        Input Spark DataFrame to profile

    Returns
    -------
    DataFrame
        A single-row Spark DataFrame where each colum contains the number of NULL values 
        found in the corresponding input column
    """

    return df.select(
        [
            count(when(col(column).isNull(), column)).alias(column)
            for column in df.columns
        ]
    )


In [12]:
def fill_missing_string_values(
        df: DataFrame
) -> DataFrame:
    """
    Fill missing values in selected string columns using
    predefined replacement values.

    Parameters
    ----------
    df : DataFrame
        Input Spark DataFrame containing transaction records

    
    Returns
    -------
    DataFrame
        Spark DataFrame with missing string value replaced
    """

    fill_values = {
        "customer_name": "Unknown",
        "customer_address": "Unknown",
        "customer_city": "Unknown",
        "customer_state": "Unknown",
        "customer_country": "Unknown",
        "company": "Unknown",
        "job_title": "Unknown",
        "email": "Unknown",
        "phone_number": "Unknown",
        "category": "Unknown",
        "group": "Unknown",
        "description": "Unknown",
        "gender": "Unknown",
        "marital_status": "Unknown",
    }

    df = df.fillna(fill_values)

    return df

In [13]:
# def standardize_column_names(
#         df: DataFrame,
# )-> DataFrame:
#     """
#     Standardise all columns names

#     This function:
#     1. Removes leading/trailing spaces.
#     2. Converts names to lowercase.
#     3. Replaces spaces with underscores.

#     Parameters
#     ----------
#     df: DataFrame
#         input Spark DataFrame

#     Returns
#     -------
#     DataFrame
#         DataFrame with standardized columns
#     """

#     for column in df.columns:

#         new_column = (
#             column
#             .strip()
#             .lower()
#             .replace(" ", "_")
#         )

#         df = df.withColumnRenamed(
#             column,
#             new_column
#         )
    
#     return df



# nuga_df = standardize_column_names(nuga_df)


# nuga_df.columns


# ///////////////////////////////////--Let's Build the Production Version--//////////////////////////////////////////////
import re

def standardize_column_names(
    df:DataFrame
)-> DataFrame:
    """
    Standardize all DataFrame column names.

    This function:
    1. Removes leading and trailing whitespace.
    2. Converts names to lowercase.
    3. Replaces spaces with underscores.
    4. Removes non-alphanumeric characters (except underscores).
    5. Detects duplicate names created during standardization.

    Parameters
    ----------
    df : DataFrame
        Input Spark DataFrame.

    Returns
    -------
    DataFrame
        DataFrame with standardized column names.

    Raises
    ------
    ValueError
        If duplicate column names are produced.
    """
        
    standardized_columns = [
        re.sub(r"[^a-z0-9_]", "", column.strip().lower().replace(" ", "_"))
        for column in df.columns
    ]


    duplicates = {
        column
        for column in standardized_columns
        if standardized_columns.count(column) > 1
    }

    if duplicates:
        raise ValueError(
            f"Duplicate column names after standardization: {sorted(duplicates)}"
        )

    return df.toDF(*standardized_columns)




In [14]:
# Verify the cleaning (A professional Engineer never assumes a transformation worked, we verify it.)
nuga_df = fill_missing_string_values(nuga_df)

summarize_missing_values(nuga_df).show(vertical=True)

-RECORD 0--------------------
 Transaction_Date   | 0      
 Amount             | 0      
 Transaction_Type   | 0      
 Customer_Name      | 0      
 Customer_Address   | 0      
 Customer_City      | 0      
 Customer_State     | 0      
 Customer_Country   | 0      
 Company            | 0      
 Job_Title          | 0      
 Email              | 0      
 Phone_Number       | 0      
 Credit_Card_Number | 100085 
 IBAN               | 100300 
 Currency_Code      | 99342  
 Random_Number      | 99913  
 Category           | 0      
 Group              | 0      
 Is_Active          | 100259 
 Last_Updated       | 100321 
 Description        | 0      
 Gender             | 0      
 Marital_Status     | 0      



In [15]:
# A better Way to Verify
# Instead of checking every column manually, lets create another reusable function

def display_dataframe_info(df: DataFrame) -> None:
    """
    Parameters
    ----------
    df: DataFrame
        Input Spark DataFrame


    Returns
    -------
    None
    """

    print("=" * 70)
    print("DataFrame Summary")
    print("=" * 70)

    print(f"Rows    : {df.count():,}")
    print(f"Columns : {len(df.columns)}")

    print("\nSchema")
    df.printSchema()


display_dataframe_info(nuga_df)

DataFrame Summary
Rows    : 1,000,000
Columns : 23

Schema
root
 |-- Transaction_Date: timestamp (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Transaction_Type: string (nullable = true)
 |-- Customer_Name: string (nullable = false)
 |-- Customer_Address: string (nullable = false)
 |-- Customer_City: string (nullable = false)
 |-- Customer_State: string (nullable = false)
 |-- Customer_Country: string (nullable = false)
 |-- Company: string (nullable = false)
 |-- Job_Title: string (nullable = false)
 |-- Email: string (nullable = false)
 |-- Phone_Number: string (nullable = false)
 |-- Credit_Card_Number: long (nullable = true)
 |-- IBAN: string (nullable = true)
 |-- Currency_Code: string (nullable = true)
 |-- Random_Number: double (nullable = true)
 |-- Category: string (nullable = false)
 |-- Group: string (nullable = false)
 |-- Is_Active: string (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)
 |-- Description: string (nullable = false)
 |-- Gender:

##### Drop Duplicate

In [16]:
def count_duplicate_rows(df: DataFrame)->int:
    """
    Count the number of exact duplicates rows in a Spark DataFrame

    Parameter
    ---------
    df: DataFrame
        input Spark DataFrame

    Returns
    -------
    int
        Number of duplicate rows
    """

    total_rows = df.count()
    unique_rows = df.drop_duplicates().count()

    return total_rows - unique_rows

In [17]:
duplicate_count = count_duplicate_rows(nuga_df)

print(f"Duplicate rows: {duplicate_count:,}")

Duplicate rows: 0


In [18]:
# Inspect the Duplicates
# Counting duplicates is useful, but seeing them is even more valuable.

def show_duplicates_rows(df: DataFrame, limit: int = 10) -> None:
    """
    Display a sample of duplicate rows.

    Parameters
    ----------
    df : DataFrame
    Input Spark DataFrame.


    limit : int, optional
        Maximum number of duplicates to display
        Defaults to 10.

    Returns
    -------
    None
    """

    (
        df.groupBy(df.columns)
        .count()
        .filter("count > 1")
        .show(limit, truncate=False)
    )

In [19]:
show_duplicates_rows(nuga_df)

+----------------+------+----------------+-------------+----------------+-------------+--------------+----------------+-------+---------+-----+------------+------------------+----+-------------+-------------+--------+-----+---------+------------+-----------+------+--------------+-----+
|Transaction_Date|Amount|Transaction_Type|Customer_Name|Customer_Address|Customer_City|Customer_State|Customer_Country|Company|Job_Title|Email|Phone_Number|Credit_Card_Number|IBAN|Currency_Code|Random_Number|Category|Group|Is_Active|Last_Updated|Description|Gender|Marital_Status|count|
+----------------+------+----------------+-------------+----------------+-------------+--------------+----------------+-------+---------+-----+------------+------------------+----+-------------+-------------+--------+-----+---------+------------+-----------+------+--------------+-----+
+----------------+------+----------------+-------------+----------------+-------------+--------------+----------------+-------+---------+--

In [20]:
# Data Type Standardization

# Before we change any data, let's inspect it.
nuga_df.groupBy("is_Active").count().show()

+---------+------+
|is_Active| count|
+---------+------+
|     NULL|100259|
|       No|449899|
|      Yes|449842|
+---------+------+



In [21]:
def convert_yes_no_to_boolean(
        df: DataFrame,
        column_name: str
) -> DataFrame:
    """
    Convert a Yes/No string column to Boolean.

    Parameter
    ---------
    df : DataFrame
        input spark DataFrame.

    column_name: str
        Name of the column to convert.

    Returns
    -------
    DataFrame
        New DataFrame with the specified column converted
        to Boolean values
    """

    return df.withColumn(
        column_name,
        when(col(column_name) == "Yes", True)
        .when(col(column_name) == "No", False)
        .otherwise(None)
    )


nuga_df = convert_yes_no_to_boolean(
    nuga_df, "is_Active"
)

nuga_df.printSchema()



root
 |-- Transaction_Date: timestamp (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Transaction_Type: string (nullable = true)
 |-- Customer_Name: string (nullable = false)
 |-- Customer_Address: string (nullable = false)
 |-- Customer_City: string (nullable = false)
 |-- Customer_State: string (nullable = false)
 |-- Customer_Country: string (nullable = false)
 |-- Company: string (nullable = false)
 |-- Job_Title: string (nullable = false)
 |-- Email: string (nullable = false)
 |-- Phone_Number: string (nullable = false)
 |-- Credit_Card_Number: long (nullable = true)
 |-- IBAN: string (nullable = true)
 |-- Currency_Code: string (nullable = true)
 |-- Random_Number: double (nullable = true)
 |-- Category: string (nullable = false)
 |-- Group: string (nullable = false)
 |-- is_Active: boolean (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)
 |-- Description: string (nullable = false)
 |-- Gender: string (nullable = false)
 |-- Marital_Status: string (nu

In [22]:
# Lets inspect the values

nuga_df.groupBy("is_Active").count().show()

+---------+------+
|is_Active| count|
+---------+------+
|     NULL|100259|
|     true|449842|
|    false|449899|
+---------+------+



#### N/B since there is no duplicate, no need to run the below syntax
nuga_df = nuga.dropDuplicates()

### Date and Timestamp Cleaning


##### Since inferschema already converted Transactio_Date from string to Timestamp which is correct, no need to do it again.

### SECTION 7: Data Normalisation / Transformation

In [23]:
# Function to trim columns

from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

def trim_string_columns(df: DataFrame) -> DataFrame:
    """
    Remove leading and trailing whitespace from every
    string column in a spark DataFrame.

    Parameters
    ----------
    df: DataFrame
        input spark DataFrame.

    Returns
    -------
    DataFrame
        DataFrame with all string columns trimmed.
    """
    expressions = []

    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            expressions.append(
                trim(col(field.name)).alias(field.name)
            )
        else:
            expressions.append(
                col(field.name)
            )
        
    return df.select(*expressions)

nuga_df = trim_string_columns(nuga_df)

## Validation

In [24]:
# validation 1: validate required columns

from typing import List

def validate_required_columns(
        df: DataFrame,
        required_columns: List[str]
)->None: # We are returning None because this function does not transform data, its job is to stop the pipeline if something is wrong.
    
    
    """
    Validate that all required columns exists in the DataFrame

    Parameter
    ---------
    df: DataFrame
        input Spark DataFrame

    required_columns : List[str]
        List of expected column names.


    Raises
    ------
    ValueError
        If one or more required columns are missing.
    """

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {', '.join(missing_columns)}"
        )
    





In [25]:
# from pyspark.sql.types import NumericType
# def validate_numeric_column(
#     df: DataFrame,
#     column_name: str
# )-> bool:
    
#     """
#     Validate that a column has a numeric Spark data type.

#     Parameters
#     ----------
#     df : DataFrame
#         Input Spark DataFrame.

#     column_name : str
#         Name of the column to validate.

#     Returns
#     -------
#     bool
#         True if the column is numeric.

#     Raises
#     ------
#     ValueError
#         If the column does not exist.
#     """

#     if column_name not in df.columns:
#         raise ValueError(f"Column {column_name} does not exist")
    
#     data_type = df.schema[column_name].dataType

#     return isinstance(data_type, NumericType)

#     if validate_numeric_column(df, "amount"):
#         print("Proceed with numeric calculations.")
#     else:
#         print("Column must be converted first.")



In [26]:
def validate_numeric_content(
        df: DataFrame,
        column_name: str
)-> None:
    
    """
    Check that all non-values in column are numeric using regex

    Parameter
    ---------
    df: DataFrame
        input Spark DataFrame

    Column_name: str
        name of column to validate

    Return:
    -------
       None

    Raises
    ------
    ValueError
        If column does not exists or contains non-numeric values
    """

    if column_name not in df.columns:
        # raise ValueError(f"column '{column_name}' does not exist")
        error_report = (
            f"column '{column_name}' does not exist"
        )
    
    NUMERIC_PATTERN  = r"^-?\d+(\.\d+)?$"

    invalid_rows = df.filter(
        col(column_name).isNotNull() &
        (~col(column_name).rlike(NUMERIC_PATTERN))
    )

    invalid_count = invalid_rows.count()

    # if invalid_count > 0:
    #     raise ValueError(
    #         f"column '{column_name}' contains {invalid_count} non-numeric values(s)."
    #     )

    if invalid_count > 0:

        error_report = (
            f"column '{column_name}' contains {invalid_count} non-numeric values(s)."
        )

        raise ValueError(error_report)
    




In [27]:
def validate_column_types(
    df: DataFrame,
    expected_schema: dict[str, DataType]
)-> None:
    """
    Validate that the DataFrame contains the expected columns and
    that each column has the expected Spark data type.

    Parameters
    ----------
    df : DataFrame
        Input Spark DataFrame.

    expected_schema : dict[str, DataType]
        Dictionary mapping column names to their expected Spark
        data types.

    Raises
    ------
    ValueError
        If one or more expected columns are missing or if one or
        more columns have an unexpected data type.

    Returns
    -------
    None
        Returns None if all columns exist and their data types
        match the expected schema.
    """
    validation_errors = []

    for column_name, expected_type in expected_schema.items():

        # check whether the column exists
        if column_name not in df.columns:
            validation_errors.append(
                f"- Missing column: '{column_name}'"
            )
            continue

        actual_type = df.schema[column_name].dataType

        # Compare Spark data types
        if type(actual_type) != type(expected_type):
            validation_errors.append(
                f"- Column '{column_name}': "
                f"expected {expected_type.simpleString()}, "
                f"found {actual_type.simpleString()}"
            )

    if validation_errors:
        error_report = (
            "schema validation failed.\n\n"
            + "\n".join(validation_errors)
        )

        raise ValueError(error_report)

In [28]:
from typing import Sequence
def validate_allowed_values(
        df: DataFrame,
        column_name: str,
        allowed_values: Sequence[str]
)-> None:
    """
    Validate that all non-values in a column belong to a predefined list of allowed values

    Parameter
    ---------
    df: DataFrame
        input Spark DataFrame

    column_name: str 
        name of column to validate

    allowed_values: Sequence[str]
        List of allowed values in the column

    
    Raises
    ------
    ValueError
        If the column does not exist or contains one or more values outside the allowed_values list

    
    Return
    ------
    None
        ReturnsNone if validation passes.    
    
    """

    #check if column exist
    if column_name not in df.columns:
        raise ValueError(
            f" - Missing column '{column_name}' does not exist"
        )
    
    # Filter rows where value is not NULL
    invalid_rows = df.filter(
        col(column_name).isNotNull() 
        & (~col(column_name).isin(allowed_values))
    )

    # count invalid rows
    invalid_count = invalid_rows.count()

    # validation failed
    if invalid_count > 0:
        # print(f"Sample invalid calues found in '{column_name}':")

        invalid_values = [
            row[column_name]
            for row in invalid_rows
            .select(column_name)
            .distinct()
            .collect() # I used collect Because this function is not responsible for displaying data.
        ]

        # raise ValueError(
        #     f"Validation failed: Column '{column_name}' "
        #     f"Found {invalid_count} invalid value(s). "
        #     f"Invalid values: {invalid_values}."
        #     f"Allowed values are: {list(allowed_values)}"
        # )

        error_report = (
            f"Validation failed: Column '{column_name}' "
            f"Found {invalid_count} invalid value(s). "
            f"Invalid values: {invalid_values}."
            f"Allowed values are: {list(allowed_values)}"
        )

        raise ValueError(error_report)

    


In [29]:
def validate_unique_keys(
        df: DataFrame,
        column_name: str
)-> None:

    """
    Parameters
    ----------
    df: DataFrame
        input Spark DataFrame
    
    column_name: str
        Name of the column to validate

    
    Raises
    ------
    ValueError
        if the column does not exist or Duplicate values are detected

        
    Returns
    -------
    None
        Returns None when all values are unique.
    """

    if column_name not in df.columns:
        raise ValueError(
            f"- Missing column '{column_name}' does not exist"
        )


    duplicate_keys = (
        df.filter(col(column_name).isNotNull())
        .groupBy(column_name)
        .count()
        .filter(col("count") > 1)
        .orderBy(col("count").desc())
    )


    duplicate_count = duplicate_keys.count()

    if duplicate_count > 0:        

        sample_duplicates = duplicate_keys.limit(5).collect()

        sample_report = ", ".join(
            f"{row[column_name]} ({row['count']} occurences)"
            for row in sample_duplicates
        )


        # raise ValueError(
        #     f"Validation Failed: Found {duplicate_count} duplicated "
            
        #     f"key(s) in column '{column_name}'. "
        #     f"sample duplicate keys: {sample_report}"
        # )

        error_report = (
            f"Validation Failed: Found {duplicate_count} duplicated "
            
            f"key(s) in column '{column_name}'. "
            f"sample duplicate keys: {sample_report}"
        )   

        raise ValueError(error_report)

In [30]:
def validate_numeric_ranges(
    df: DataFrame,
    column_name: str,
    minimum: float, # we use float to keep it reusable
    maximum: float
) -> None:
    """
        Validate that all non-null numeric values in a column fall
    within a specified inclusive range.

    Parameters
    ----------
    df : DataFrame
        Input Spark DataFrame.

    column_name : str
        Name of the numeric column to validate.

    minimum : float
        Minimum acceptable value (inclusive).

    maximum : float
        Maximum acceptable value (inclusive).

    Raises
    ------
    ValueError
        If the column does not exist or contains values outside
        the specified numeric range.

    Returns
    -------
    None
        Returns None if validation passes.
    """

    if column_name not in df.columns:
        # raise ValueError(
        #     f" - Missing Column '{column_name}' does not exist"
        # )
        error_report = (
            f" - Missing Column '{column_name}' does not exist"
        )
    if minimum > maximum:
        # raise ValueError(
        #     "Minimum cannot be greater than maximum"
        # )
        error_report = (
            f" - Invalid range: Minimum cannot be greater than maximum"
        )

    invalid_rows = (
        df.filter(
            col(column_name).isNotNull()
            &
            (
                (col(column_name) < minimum)
                |
                (col(column_name) > maximum)
            )
        )
    )

    invalid_count = invalid_rows.count()

    if invalid_count > 0:

        sample_invalid_values = [
            row[column_name]
            for row in invalid_rows
                .select(column_name)
                .distinct()
                .limit()
                .collect()
        ]

        # raise ValueError(
        #     f"Validate failed: Found {invalid_count} value(s) "
        #     f"outside the allowed range "
        #     f"[{minimum}, {maximum}] "
        #     f"Sample values: {sample_invalid_values}"
        # )
        error_report = (
            f"Validate failed: Found {invalid_count} value(s) "
            f"outside the allowed range "
            f"[{minimum}, {maximum}] "
            f"Sample values: {sample_invalid_values}"
        )

        raise ValueError(error_report)



# Data Normalisation | Data Modeling

In [31]:
# Creating the Customer Dimesion

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

def create_customer_dimension(
    df: DataFrame,
    customer_columns: list[str]
)->DataFrame:

    """
    Parameters
    ---------
    df: DataFrame
        input Spark DataFrame

    customer_columns
        list of customer columns used to construct the customer dimension

    Raises
    -----
    ValueError
        if column one or more cuctomer columns do not exist in the DataFrame

    Returns
    ------
        Spark DataFrame containing unique customer records with a generated 
        customer_id surrogate key

    """
    # validate required columns
    validate_required_columns(  # we have previously written this function, so we are calling it here.
        df, customer_columns
    )

    # select only customer columns
    customers_df = df.select(*customer_columns)

    # Remove exact duplicate customer records
    customers_df = customers_df.dropDuplicates(customer_columns)

    # Generating the surrogate Key
    window_spec = Window.orderBy(
        "customer_name", 
        "email"
    )
    customers_df = customers_df.withColumn( # -> withColumn() adds a new column 
        "customer_id",
        row_number().over(window_spec) # -> row_number() generates sequential numbers | .over() applies the ordering rules
    )

    return customers_df




In [32]:
# Create fact dimension df

from functools import reduce


def create_transactions_fact(
    transactions_df: DataFrame,
    customers_df: DataFrame,
    fact_columns: list[str],
    join_columns: list[str]
) -> DataFrame:

    """
    Create a transaction fact table by joining the transaction
    DataFrame with the customer dimension.

    Parameters
    ----------
    transactions_df : DataFrame
        Cleaned transaction DataFrame.

    customers_df : DataFrame
        Customer dimension containing the surrogate customer_id.

    fact_columns : list[str]
        Transaction columns to include in the fact table.

    join_columns : list[str]
        Columns used to join the transaction DataFrame with the
        customer dimension.

    Raises
    ------
    ValueError
        If one or more required columns are missing.

    Returns
    -------
    DataFrame
        Transaction fact table.
    """

    # ------------------------------------------
    # Validate required columns
    # ------------------------------------------

    validate_required_columns(
        transactions_df,
        fact_columns + join_columns
    )

    validate_required_columns(
        customers_df,
        ["customer_id"] + join_columns
    )

    # ------------------------------------------
    # Create aliases
    # ------------------------------------------

    t = transactions_df.alias("t")
    c = customers_df.alias("c")

    # ------------------------------------------
    # Build the join condition dynamically
    # ------------------------------------------

    join_condition = reduce(
        lambda condition1, condition2: condition1 & condition2,
        [
            col(f"t.{column}") == col(f"c.{column}")
            for column in join_columns
        ]
    )

    # ------------------------------------------
    # Join transaction data with customer dimension
    # ------------------------------------------

    joined_df = t.join(
        c,
        join_condition,
        how="left"
    )

    # ------------------------------------------
    # Build the list of columns to select
    # ------------------------------------------

    selected_columns = [
        col("c.customer_id")
    ]

    selected_columns.extend(
        col(f"t.{column}")
        for column in fact_columns
    )

    # ------------------------------------------
    # Create fact table
    # ------------------------------------------

    fact_transactions_df = joined_df.select(
        *selected_columns
    )

    return fact_transactions_df

## Loading Data Into PostgreSQL

In [33]:
def load_to_postgres(
    df: DataFrame,
    table_name: str,
    jdbc_url: str,
    username: str,
    password: str,
    mode: str = "append",
    driver: str = "org.postgresql.Driver"
)-> None:
    """
    Load a Spark DataFrame into a PostgreSQL table.

    Parameters
    ----------
    df : DataFrame
        Input Spark DataFrame to load.

    table_name : str
        Name of the target PostgreSQL table.

    jdbc_url : str
        JDBC URL for the PostgreSQL database.

    username : str
        Username for the PostgreSQL database.

    password : str
        Password for the PostgreSQL database.

    mode : str, optional
        Save mode (default is "append"). Options include:
        "append", "overwrite", "ignore", "error".

    driver : str, optional
        JDBC driver class name (default is "org.postgresql.Driver").

    Returns
    -------
    None
        This function does not return any value.
    """

    valid_modes = {
        "append",
        "overwrite",
        "ignore",
        "error",
    }

    if mode not in valid_modes:
        raise ValueError(
            f"Invalid write mode '{mode}'." 
            f"choose one of {sorted(valid_modes)}"
        )
       

    try:
        (
            df.write
            .format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", table_name)
            .option("user", username)
            .option("password", password)
            .option("driver", driver)
            .mode(mode)
            .save()
        )
    except Exception as error:
        raise RuntimeError(
            f"Failed to load DataFrame into PostgreSQL table '{table_name}': {error}"
        ) from error
        

    # # Log a message indicating successful loading | best used for production pipelines
    # logger.info(
    #     f"DataFrame successfully loaded into PostgreSQL table '{table_name}' using mode '{mode}'"
    # )

    print(
        f"DataFrame successfully loaded into PostgreSQL table '{table_name}' using mode '{mode}'"
    )

# Ochestration Functions

### Clean data ochestration

In [34]:
def clean_data(
        df: DataFrame,
        yes_no_columns : str,
)-> DataFrame:
    """
    Apply all data cleaning transformations in the correct order.

    Parameters
    ----------
    df : DataFrame
        Input Spark DataFrame to clean.

    yes_no_columns : list[str]
        Columns containing "Yes"/"No" values to convert to Boolean.

    Returns
    -------
    DataFrame
        Cleaned Spark DataFrame.
    """

    # 1 Standardize column names
    df = standardize_column_names(df)

    # 2 fill missing string values
    df = fill_missing_string_values(df)

    # 3 Remove leading/trailing whitespace from string columns
    df = trim_string_columns(df)

    # 4 Convert Yes/No columns to Boolean
    df = convert_yes_no_to_boolean(
        df, 
        yes_no_columns
        )

    # Additional cleaning functions depending on your dataset can be added here. For example:
    # df = clean_transaction_type(...)
    # df = convert_transaction_date(...)
    # df = clean_currency_code(...)
    # df = ...

    return df

### testing the cleaning orchestration....

In [35]:
cleaned_df = clean_data(
    nuga_df,
    yes_no_columns="is_active"
)

cleaned_df.printSchema()

root
 |-- transaction_date: timestamp (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- customer_address: string (nullable = false)
 |-- customer_city: string (nullable = false)
 |-- customer_state: string (nullable = false)
 |-- customer_country: string (nullable = false)
 |-- company: string (nullable = false)
 |-- job_title: string (nullable = false)
 |-- email: string (nullable = false)
 |-- phone_number: string (nullable = false)
 |-- credit_card_number: long (nullable = true)
 |-- iban: string (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- random_number: double (nullable = true)
 |-- category: string (nullable = false)
 |-- group: string (nullable = false)
 |-- is_active: boolean (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- description: string (nullable = false)
 |-- gender: string (nullable = false)
 |-- marital_status: string (nu

In [36]:
cleaned_df.show(truncate=False)

+--------------------------+------+----------------+-----------------+-----------------------------+----------------------+--------------+--------------------------------------------+--------------------------------+-------------------------------------------+----------------------------+-------------------+-------------------+----------------------+-------------+-------------+--------+-------+---------+--------------------------+-------------------------------------------------+-------+--------------+
|transaction_date          |amount|transaction_type|customer_name    |customer_address             |customer_city         |customer_state|customer_country                            |company                         |job_title                                  |email                       |phone_number       |credit_card_number |iban                  |currency_code|random_number|category|group  |is_active|last_updated              |description                                      |gender |

## validation ochestration

In [37]:
from typing import Optional

def  validate_data(
    df: DataFrame,
    required_columns: list[str],
    expected_schema: dict[str, DataType],
    allowed_values_rules: dict[str, Sequence[str]],
    unique_key_column: Optional[str],
    numeric_range_column: Optional[str],
    minimum: Optional[float],
    maximum: Optional[float]
)-> None:

    validation_errors = []

    try:
        validate_required_columns(
            df, 
            required_columns
        )

    except ValueError as error:
        validation_errors.append(
            str(error)
        )

    try:
        validate_column_types(
            df,
            expected_schema
        )

    except ValueError as error:
        validation_errors.append(
            str(error)
        )
# validate all allowed values in the DataFrame using the provided rules
    for column_name, allowed_values in allowed_values_rules.items():

        try:
            validate_allowed_values(
                df,
                column_name,
                allowed_values
            )
        except ValueError as error:
            validation_errors.append(
                str(error)
            )


    # try:
    #     validate_unique_keys(
    #         df,
    #         unique_key_column
    #     )
    # except ValueError as error:
    #     validation_errors.append(
    #         str(error)
    #     )
    if unique_key_column is not None:
        try:
            validate_unique_keys(
                df,
                unique_key_column
            )
        except ValueError as error:
            validation_errors.append(
                str(error)
            )

    try:
        validate_numeric_ranges(
            df,
            numeric_range_column,
            minimum,
            maximum
        )
    except ValueError as error:
        validation_errors.append(
            str(error)
        )


    # Raise all validation errors after
    # every validation check has been performed
    if validation_errors:
        error_report = (
            "Data Validation failed:\n"
            + "\n".join(
                f"- {error}"
                for error in validation_errors
            )
        )

        raise ValueError(error_report)

    return None


### testing the validation orchestration....

In [38]:
cleaned_df.printSchema()

root
 |-- transaction_date: timestamp (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- customer_address: string (nullable = false)
 |-- customer_city: string (nullable = false)
 |-- customer_state: string (nullable = false)
 |-- customer_country: string (nullable = false)
 |-- company: string (nullable = false)
 |-- job_title: string (nullable = false)
 |-- email: string (nullable = false)
 |-- phone_number: string (nullable = false)
 |-- credit_card_number: long (nullable = true)
 |-- iban: string (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- random_number: double (nullable = true)
 |-- category: string (nullable = false)
 |-- group: string (nullable = false)
 |-- is_active: boolean (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- description: string (nullable = false)
 |-- gender: string (nullable = false)
 |-- marital_status: string (nu

In [39]:
cleaned_df.show(truncate=False)

+--------------------------+------+----------------+-----------------+-----------------------------+----------------------+--------------+--------------------------------------------+--------------------------------+-------------------------------------------+----------------------------+-------------------+-------------------+----------------------+-------------+-------------+--------+-------+---------+--------------------------+-------------------------------------------------+-------+--------------+
|transaction_date          |amount|transaction_type|customer_name    |customer_address             |customer_city         |customer_state|customer_country                            |company                         |job_title                                  |email                       |phone_number       |credit_card_number |iban                  |currency_code|random_number|category|group  |is_active|last_updated              |description                                      |gender |


The actual data also confirms transaction types such as Withdrawal, Deposit, and Transfer, so we now have evidence for those values rather than guessing.

For this project, I recommend validating all columns, because we have a known cleaned schema.

## Validation configuration/testing setup

In [40]:
expected_schema = {
    "transaction_date": TimestampType(),
    "amount": DoubleType(),
    "transaction_type": StringType(),
    "customer_name": StringType(),
    "customer_address": StringType(),
    "customer_city": StringType(),
    "customer_state": StringType(),
    "customer_country": StringType(),
    "company": StringType(),
    "job_title": StringType(),
    "email": StringType(),
    "phone_number": StringType(),
    "credit_card_number": LongType(),
    "iban": StringType(),
    "currency_code": StringType(),
    "random_number": DoubleType(),
    "category": StringType(),
    "group": StringType(),
    "is_active": BooleanType(),
    "last_updated": TimestampType(),
    "description": StringType(),
    "gender": StringType(),
    "marital_status": StringType()
}

required_columns = list(expected_schema.keys())


allowed_values_rules = {
    "transaction_type":[
    "Deposit",
    "Transfer",
    "Withdrawal"
    ],

    "category":[
        "A",
        "B",
        "C",
        "D",
        "Unknown"
    ],
    "gender":[
        "Female",
        "Male",
        "Other",
        "Unknown"
    ]
}


numeric_range_column = "amount"
minimum = 10.0
maximum = 1000.0
unique_key_column = None  # Set to None if there is no unique key column


In [41]:
# Now we can test the actual NugaBank configuration

validate_data(
    cleaned_df,
    required_columns=required_columns,
    expected_schema=expected_schema,
    allowed_values_rules=allowed_values_rules,
    unique_key_column=unique_key_column,
    numeric_range_column=numeric_range_column,
    minimum=minimum,
    maximum=maximum
)

# Building the Customer Dimension

In [49]:
customer_columns = [
    "customer_name",
    "customer_address",
    "customer_city",
    "customer_state",
    "customer_country",
    "company",
    "job_title",
    "email",
    "phone_number",
    "credit_card_number",
    "iban",
    "gender",
    "marital_status"
]

In [50]:
customer_dim_df = create_customer_dimension(
    cleaned_df,
    customer_columns
)



In [51]:
customer_dim_df.printSchema()

root
 |-- customer_name: string (nullable = false)
 |-- customer_address: string (nullable = false)
 |-- customer_city: string (nullable = false)
 |-- customer_state: string (nullable = false)
 |-- customer_country: string (nullable = false)
 |-- company: string (nullable = false)
 |-- job_title: string (nullable = false)
 |-- email: string (nullable = false)
 |-- phone_number: string (nullable = false)
 |-- credit_card_number: long (nullable = true)
 |-- iban: string (nullable = true)
 |-- gender: string (nullable = false)
 |-- marital_status: string (nullable = false)
 |-- customer_id: integer (nullable = false)



In [52]:
customer_dim_df.show(truncate=False)

+---------------+-----------------------------------+-----------------+--------------+--------------------------------------------+----------------------------+----------------------------------+------------------------------+----------------------+------------------+----------------------+-------+--------------+-----------+
|customer_name  |customer_address                   |customer_city    |customer_state|customer_country                            |company                     |job_title                         |email                         |phone_number          |credit_card_number|iban                  |gender |marital_status|customer_id|
+---------------+-----------------------------------+-----------------+--------------+--------------------------------------------+----------------------------+----------------------------------+------------------------------+----------------------+------------------+----------------------+-------+--------------+-----------+
|Aaron Abbott   |87

We previously decided to use:

customer_name + email

before we blindly build the fact table, we should test the uniqueness of our proposed join key

In [ ]:
customer_dim_df.groupBy(
    "customer_name",
    "email"
).agg(
    count("*").alias("record_count")
).filter(
    col("record_count") > 1
).show(truncate=False)
"""
This test has revealed an important issue, and I'm glad we checked before building the fact table.

My proposed join key:

(customer_name + email) # is not unique in customer_dim_df.

So what should we do? We can either: 

A. Use another natural key, such as (email + phone_number) or (email + phone_number + customer_name) if they are unique. We should test rather than assume.

B. create a proper customer/business identifier earlier.

C. Accept the generated customer dimension represents customer records rather than true people, and that the fact table will have multiple records for the same person if they have multiple records in the source data.
"""

+---------------+--------------------+------------+
|customer_name  |email               |record_count|
+---------------+--------------------+------------+
|Robert Norris  |Unknown             |3           |
|Brittany Morris|Unknown             |2           |
|Michael Shelton|Unknown             |4           |
|Melanie Cook   |Unknown             |2           |
|Kevin Turner   |Unknown             |2           |
|Joseph Sullivan|Unknown             |3           |
|Unknown        |amanda54@example.org|2           |
|Unknown        |obrown@example.org  |5           |
|Unknown        |ncarr@example.com   |2           |
|Daniel Perez   |Unknown             |5           |
|Corey Jones    |Unknown             |4           |
|John Phillips  |Unknown             |4           |
|Austin Davis   |Unknown             |3           |
|Jose Thompson  |Unknown             |3           |
|Taylor Martin  |Unknown             |3           |
|Unknown        |qjones@example.com  |5           |
|Mark Miller

In [55]:
# Lets continue the investigation by checking the uniqueness of other potential join keys.

customer_dim_df.groupBy(
    "email",
    "phone_number"
).agg(
    count("*").alias("record_count")
).filter(
    col("record_count")>1
).show(truncate=False)

+---------------------------+------------+------------+
|email                      |phone_number|record_count|
+---------------------------+------------+------------+
|watkinslisa@example.org    |Unknown     |2           |
|smithnathan@example.net    |Unknown     |2           |
|melanie16@example.net      |Unknown     |2           |
|rebecca54@example.com      |Unknown     |2           |
|edward36@example.com       |Unknown     |2           |
|pdavis@example.net         |Unknown     |3           |
|angela88@example.com       |Unknown     |2           |
|jennifersmith@example.com  |Unknown     |7           |
|thomas10@example.org       |Unknown     |2           |
|qmartinez@example.org      |Unknown     |3           |
|wwilliams@example.com      |Unknown     |6           |
|mbrown@example.com         |Unknown     |6           |
|dennisschroeder@example.net|Unknown     |2           |
|ryan44@example.com         |Unknown     |2           |
|cody42@example.com         |Unknown     |2     

In [56]:
customer_dim_df.groupBy(
    "customer_name",
    "phone_number"
).agg(
    count("*").alias("record_count")
).filter(
    col("record_count") > 1
).show(truncate=False)

+----------------+------------+------------+
|customer_name   |phone_number|record_count|
+----------------+------------+------------+
|Daniel Robinson |Unknown     |2           |
|James Smith     |Unknown     |29          |
|Corey Jones     |Unknown     |2           |
|Kevin Eaton     |Unknown     |2           |
|Michael Hudson  |Unknown     |3           |
|Julie Stewart   |Unknown     |3           |
|Aaron Reynolds  |Unknown     |2           |
|John Phillips   |Unknown     |3           |
|Robert Villa    |Unknown     |2           |
|Shane Brown     |Unknown     |2           |
|Thomas Rivera   |Unknown     |2           |
|Karen Arnold    |Unknown     |2           |
|Thomas Taylor   |Unknown     |3           |
|Melissa Bradshaw|Unknown     |3           |
|Charles Paul    |Unknown     |2           |
|Karen Taylor    |Unknown     |5           |
|Nicholas Mendez |Unknown     |2           |
|Sean Lee        |Unknown     |3           |
|Tammy Smith     |Unknown     |10          |
|John Cook

In [57]:
customer_dim_df.groupBy(
    "customer_name",
    "email",
    "phone_number"
).agg(
    count("*").alias("record_count")
).filter(
    col("record_count") > 1
).show(truncate=False)

+-----------------+----------------------+------------+------------+
|customer_name    |email                 |phone_number|record_count|
+-----------------+----------------------+------------+------------+
|Unknown          |mark98@example.net    |Unknown     |2           |
|Karen Thompson   |Unknown               |Unknown     |2           |
|Unknown          |wbarnes@example.org   |Unknown     |2           |
|Christine White  |Unknown               |Unknown     |2           |
|David Wright     |Unknown               |Unknown     |2           |
|Matthew Cox      |Unknown               |Unknown     |2           |
|Anthony Lewis    |Unknown               |Unknown     |3           |
|Brenda Crawford  |Unknown               |Unknown     |2           |
|Jennifer Gonzalez|Unknown               |Unknown     |2           |
|Unknown          |zwilliams@example.com |Unknown     |2           |
|Unknown          |kimberly35@example.com|Unknown     |2           |
|Robert Jones     |Unknown        

from the result we've gathered, it means we should not use any of those combinations as the join key. If all three produce duplicate records, forcing a join would risk multiplying transactions in the fact table.

More importantly, this tells us something about the source data model: the dataset does not appear to contain a reliable customer identifier that lets us map a transaction unambiguously to one customer.

## One more important point

Our customer_id generated by:

##### row_number().over(window_spec)
is a surrogate key, but it doesn't solve the matching problem.
For example:
Robert Norris → customer_id 100
Robert Norris → customer_id 101
Robert Norris → customer_id 102

The surrogate key tells us these are three separate dimension rows; it doesn't tell us which one a transaction belongs to.
That's why we need to solve the natural/business-key problem before building the fact table.

In [69]:
# customer records with atleast one identifying attribute missing
customer_dim_df.filter(
    (col("customer_name") == "Unknown") |
    (col("email") == "Unknown") |
    (col("phone_number") == "Unknown")
).count()



271911

In [70]:
customer_dim_df.count()

1000000

In [66]:
cleaned_df.select("transaction_type").distinct().show()

+----------------+
|transaction_type|
+----------------+
|         Deposit|
|        Transfer|
|      Withdrawal|
+----------------+



In [64]:
cleaned_df.select("amount").describe().show()

+-------+------------------+
|summary|            amount|
+-------+------------------+
|  count|           1000000|
|   mean|504.97371121999805|
| stddev|285.79972024412314|
|    min|              10.0|
|    max|            1000.0|
+-------+------------------+



In [65]:
cleaned_df.select("transaction_type").distinct().show()
cleaned_df.select("category").distinct().show()
cleaned_df.select("gender").distinct().show()

+----------------+
|transaction_type|
+----------------+
|         Deposit|
|        Transfer|
|      Withdrawal|
+----------------+

+--------+
|category|
+--------+
|       B|
| Unknown|
|       D|
|       C|
|       A|
+--------+

+-------+
| gender|
+-------+
| Female|
|Unknown|
|  Other|
|   Male|
+-------+



There's another important issue. 
Your actual schema has no transaction_id.
Therefore, we should not do this:

unique_key_column = "transaction_date"